# Multi-View Dataset Preparation
In this notebook, we load both the **Front** and **Lateral** camera views from the normalized datasets, group them by session to prevent data leakage, and split them stratified by class label into train/val/test sets.

In [1]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
def get_session_id(file_name):
    # Extracts the unique session identifier from the filename.
    # Filenames are like: W001S01F_01.npy or W001S01L_01.npy.
    # We replace 'F_' and 'L_' with '_' to get a unified session ID: 'W001S01_01'
    base = os.path.basename(file_name)
    unified = re.sub(r'[FL]_', '_', base)
    return unified.replace('.npy', '')

In [3]:
def split_by_session(proc_dir, out_suffix):
    samples = []
    # Walk all files recursively
    for dirpath, dirnames, filenames in os.walk(proc_dir):
        for f in filenames:
            if f.endswith(".npy"):
                parent_dir_name = os.path.basename(dirpath)
                full_path = os.path.join(dirpath, f).replace("\\", "/")
                
                # Retrieve label and unified session ID
                label = parent_dir_name
                session_id = get_session_id(f)
                
                samples.append({
                    "file": full_path,
                    "label": label,
                    "session_id": session_id
                })
                
    df = pd.DataFrame(samples)
    print(f"Total files found in {proc_dir}: {len(df)}")
    
    # Get unique sessions and their labels for splitting
    unique_sessions = df[["session_id", "label"]].drop_duplicates().reset_index(drop=True)
    print(f"Unique sessions: {len(unique_sessions)}")
    
    # Split session IDs stratified by label (70% train / 15% val / 15% test)
    train_sess, test_sess = train_test_split(unique_sessions, test_size=0.30, stratify=unique_sessions.label, random_state=42)
    val_sess, test_sess = train_test_split(test_sess, test_size=0.50, stratify=test_sess.label, random_state=42)
    
    # Create sets of session IDs for fast lookup
    train_set = set(train_sess.session_id)
    val_set = set(val_sess.session_id)
    test_set = set(test_sess.session_id)
    
    # Map back to full dataset
    train_df = df[df.session_id.isin(train_set)].copy()
    val_df = df[df.session_id.isin(val_set)].copy()
    test_df = df[df.session_id.isin(test_set)].copy()
    
    # Print splits summary
    print(f"Splits summary for {out_suffix}:")
    print(f"  Train: {len(train_df)} files ({len(train_sess)} sessions)")
    print(f"  Val:   {len(val_df)} files ({len(val_sess)} sessions)")
    print(f"  Test:  {len(test_df)} files ({len(test_sess)} sessions)")
    
    # Drop temp columns and keep file, label
    train_df = train_df[["file", "label"]]
    val_df = val_df[["file", "label"]]
    test_df = test_df[["file", "label"]]
    
    os.makedirs("../Datasets", exist_ok=True)
    train_df.to_csv(f"../Datasets/train_{out_suffix}.csv", index=False)
    val_df.to_csv(f"../Datasets/val_{out_suffix}.csv", index=False)
    test_df.to_csv(f"../Datasets/test_{out_suffix}.csv", index=False)
    print(f"Saved CSVs to Datasets/train_{out_suffix}.csv etc.\n")

In [4]:
# 1. Baseline Zero-Padded Splits
split_by_session("../Datasets/normalized_landmarks", "multiview")

# 2. Interpolated Splits
split_by_session("../Datasets/normalized_landmarks_Interpolated", "interpolated_multiview")

Total files found in ../Datasets/normalized_landmarks: 102173
Unique sessions: 60851
Splits summary for multiview:
  Train: 71621 files (42595 sessions)
  Val:   15258 files (9128 sessions)
  Test:  15294 files (9128 sessions)
Saved CSVs to Datasets/train_multiview.csv etc.

Total files found in ../Datasets/normalized_landmarks_Interpolated: 102173
Unique sessions: 60851
Splits summary for interpolated_multiview:
  Train: 71621 files (42595 sessions)
  Val:   15258 files (9128 sessions)
  Test:  15294 files (9128 sessions)
Saved CSVs to Datasets/train_interpolated_multiview.csv etc.

